# Prediction-Aware Router Pipeline

This notebook runs the chronological split, frozen-expert router input generation, prediction-aware router training, and final untouched-test evaluation through the shared implementation in `scripts/chronological_expert_training.py`. Edit that script to change the router used here.


## 1. Project Setup

Run this first. It moves the working directory to the repo root and makes `src/` importable.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

src_path = ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Repo root: {ROOT}")


## 2. Shared Pipeline Imports

The router pipeline lives in `scripts/chronological_expert_training.py`, so this notebook stays thin and uses the same code as scripts/tests.


In [ ]:
import torch

from scripts.chronological_expert_training import (
    ForecastRouter,
    PredictionAwareRouter,
    run_chronological_expert_stages,
    run_final_router_test_stage,
    run_prediction_aware_router_training_stage,
    run_router_input_and_forward_stage,
)


## 3. Shared Run Settings

Adjust these paths or batch size before running any stage cells.

In [ ]:
DATA_DIR = "datasets/ETTh1"
CHECKPOINT_DIR = "checkpoints"
RESULTS_DIR = "results/router_summary"
BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 7

print(f"Device: {DEVICE}")


## 4. Stages 1 and 2: Legacy Split Data, Train Experts

Optional legacy stage. The router stages below now use the already-trained candidate checkpoints in `checkpoints/candidates/`, so this stage is not needed unless you intentionally want to retrain the older two-expert DLinear/iTransformer pair.

In [ ]:
RUN_STAGE_1_2 = False

if RUN_STAGE_1_2:
    run_chronological_expert_stages(
        data_dir=DATA_DIR,
        output_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        max_epochs=50,
        patience=5,
        learning_rate=1e-3,
        device=DEVICE,
        seed=SEED,
    )
else:
    print("Set RUN_STAGE_1_2 = True only if you want to retrain the legacy DLinear/iTransformer experts.")


## 5. Stages 3 and 4: Frozen Expert Router Inputs and Forward Pass

Loads the candidate checkpoints selected in `scripts/router_model_config.py`, freezes them, builds router inputs from the 15% router-training split, and verifies the prediction-aware router forward shapes without training.

In [ ]:
RUN_STAGE_3_4 = False

if RUN_STAGE_3_4:
    run_router_input_and_forward_stage(
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
    )
else:
    print("Set RUN_STAGE_3_4 = True to verify frozen expert router inputs and router forward shapes.")


## 6. Stage 5: Train and Validate Router

Trains only the prediction-aware router on the 15% router-training split using the selected candidate experts as frozen experts, validates on the 5% router-validation split, and saves one checkpoint per best saved model group. With `AUTO_SELECT_BEST_BY_SIZE = True`, the one-model baseline is skipped for router training and the 2, 3, 4, and 5-model routers are trained separately.

In [ ]:
RUN_STAGE_5 = False

if RUN_STAGE_5:
    run_prediction_aware_router_training_stage(
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        max_epochs=50,
        patience=10,
        learning_rate=1e-3,
        device=DEVICE,
        seed=SEED,
    )
else:
    print("Set RUN_STAGE_5 = True to train and validate the configured router groups.")


## 7. Stage 6: Final Untouched Test

Loads the selected candidate expert checkpoints plus each matching router checkpoint, evaluates the final system once on the untouched 20% test split, and writes comparison CSV/JSON files for each configured model group.

In [ ]:
RUN_STAGE_6 = False

if RUN_STAGE_6:
    run_final_router_test_stage(
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        output_dir=RESULTS_DIR,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
    )
else:
    print("Set RUN_STAGE_6 = True to run the one-time untouched test evaluation.")
